# Cincinnati Transit Vulnerability · 01 Data ingestion

**Author:** Omid Mansourihanis
**Data:** SORTA/Metro GTFS feed, downloaded from the agency's public developer page.

```
GTFS zip  →  parse tables  →  quality checks  →  weekday service IDs
```

The raw feed is cached in `data/raw/` and is not committed to the repository.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
(DATA / "processed").mkdir(parents=True, exist_ok=True)

In [ ]:
from src import gtfs_loader

zip_path = gtfs_loader.download_gtfs(cache_path=str(DATA / "raw" / "sorta_gtfs.zip"))
gtfs = gtfs_loader.load_gtfs(zip_path)
{name: len(df) for name, df in gtfs.items()}

## Quality checks
The stop extent should fall in the Cincinnati region (about 39.0–39.4 N, 84.3–84.8 W).

In [ ]:
summary = gtfs_loader.summarize(gtfs)
summary

In [ ]:
for name in ["stops", "routes", "trips", "stop_times"]:
    print(f"{name:11s} missing values: {int(gtfs[name].isna().sum().sum()):,}")
if "route_type" in gtfs["routes"]:
    print(gtfs["routes"]["route_type"].value_counts())

## Weekday service
Service IDs are read from `calendar.txt` (services running Monday to Friday) rather than hard-coded, because agencies rename them between feed versions.

In [ ]:
weekday = gtfs_loader.weekday_service_ids(gtfs)
print("Weekday service IDs:", weekday)
import json
(DATA / "processed" / "weekday_service_ids.json").write_text(json.dumps(weekday))